<a href="https://colab.research.google.com/github/sudarshan-khot/Volumetric-Segmentation-and-Stroke-Triage-via-Transformers/blob/main/Final_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Based Dual-Pipeline for Stroke Infarct Analysis

This notebook represents the final end-to-end inference pipeline.
It combines two specialized neural networks to evaluate non-contrast CT scans:
1. **Classification (EfficientNet-V2-S)**: Rapidly scans the 2D slices to detect the presence of a stroke.
2. **Segmentation (Volumetric DynUNet)**: If a stroke is detected, it extracts 25mm volumetric Z-axis stacks (5 contiguous slices) to accurately segment the infarct penumbra and core, outputting the final infarct volume in mL.


In [ ]:
!pip install -q -U segmentation-models-pytorch albumentations pandas opencv-python-headless timm monai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 119.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3,

In [ ]:

import os
import re
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from monai.networks.nets import DynUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ System Ready. Using device: {device}")


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


✅ System Ready. Using device: cuda


## Model Initialization & Weight Loading
We instantiate our production models and load the best checkpoints from our training phase.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Initializing AI Models...")

# 1. Initialize Classifier (EfficientNet-V2-S)
classifier = models.efficientnet_v2_s(weights=None)
classifier.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(classifier.classifier[1].in_features, 1)
)

segmenter = DynUNet(
    spatial_dims=2,
    in_channels=5,
    out_channels=1,
    kernel_size=[3, 3, 3, 3, 3],
    strides=[1, 2, 2, 2, 2],
    upsample_kernel_size=[2, 2, 2, 2],
    filters=[64, 128, 256, 512, 1024],
).to(device)

# Move to device
classifier = classifier.to(device)
segmenter = segmenter.to(device)

# --- LOAD WEIGHTS ---
# Update these paths to your final checkpoints
CLS_WEIGHTS = "/content/drive/MyDrive/Stroke_Infarct_Volume_Sudarshan/checkpoints/best_classifier.pt"
SEG_WEIGHTS = "/content/drive/MyDrive/Stroke_Infarct_Volume_Sudarshan/checkpoints/best_segmenter.pt"

try:
    classifier.load_state_dict(torch.load(CLS_WEIGHTS, map_location=device))
    print("✅ Classifier weights loaded successfully.")
except Exception as e:
    print(f"⚠️ Warning: Could not load classifier weights. Check path. {e}")

try:
    checkpoint = torch.load(SEG_WEIGHTS, map_location=device)
    # Check if 'model_state_dict' wrapper exists (due to our custom save logic), otherwise load directly
    if 'model_state_dict' in checkpoint:
        segmenter.load_state_dict(checkpoint['model_state_dict'])
    else:
        segmenter.load_state_dict(checkpoint)
    print("✅ Segmenter weights loaded successfully.")
except Exception as e:
    print(f"⚠️ Warning: Could not load segmenter weights. Check path. {e}")




Initializing AI Models...
✅ Classifier weights loaded successfully.
✅ Segmenter weights loaded successfully.


In [ ]:
classifier.eval()
segmenter.eval()

DynUNet(
  (input_block): UnetBasicBlock(
    (conv1): Convolution(
      (conv): Conv2d(5, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (conv2): Convolution(
      (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
    (norm1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
    (norm2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
  )
  (downsamples): ModuleList(
    (0): UnetBasicBlock(
      (conv1): Convolution(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      )
      (conv2): Convolution(
        (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
      (norm1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track

---
## Clinical Processing Engine
This section defines the core clinical logic workflow.
- **Stage 1 (Classification)**: The CT sequence is evaluated slice-by-slice.
- **Stage 2 (Triaging)**: If the peak probability exceeds 50%, the scan is flagged `Positive for Stroke`. If not, the pipeline terminates cleanly to save compute.
- **Stage 3 (Volumetric Segmentation)**: Positive scans are converted into 5-channel volumetric tensors (z-axis sliding window) and evaluated to calculate the true physiological Infarct Volume.


In [ ]:
def apply_bone_window(image_path, W=40, L=50):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return np.zeros((256, 256), dtype=np.uint8)

    # Crucial: Resize to match training dims (256x256) solving OOM Memory errors
    img = cv2.resize(img, (256, 256))
    min_val = L - (W / 2)
    max_val = L + (W / 2)
    windowed = np.clip(img, min_val, max_val)
    windowed = ((windowed - min_val) / (max_val - min_val) * 255).astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(windowed)

def analyze_patient_ct(patient_dir):
    print("="*60)
    print(f"🔍 INITIATING ANALYSIS FOR PATIENT: {os.path.basename(patient_dir)}")
    print("="*60)

    if not os.path.exists(patient_dir):
        print(f"❌ Directory not found: {patient_dir}")
        return

    # Sort files naturally so slices are in correct Z-axis order
    image_files = sorted([f for f in os.listdir(patient_dir) if f.lower().endswith(('.png', '.jpg'))])
    if not image_files:
        print("❌ No images found in directory.")
        return

    # --- STAGE 1: CLASSIFICATION ---
    print("⏳ Stage 1: Running Rapid Classification Scan...")
    cls_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    stroke_probabilities = []

    with torch.no_grad():
        for img_name in image_files:
            img_path = os.path.join(patient_dir, img_name)
            img_rgb = Image.open(img_path).convert('RGB') # EfficientNet requires 3-channel
            tensor_img = cls_tf(img_rgb).unsqueeze(0).to(device)

            output = classifier(tensor_img)
            prob = torch.sigmoid(output).item()
            stroke_probabilities.append(prob)

    max_prob = max(stroke_probabilities)
    is_stroke = max_prob > 0.5

    if not is_stroke:
        print(f"✅ DIAGNOSIS: NORMAL (Negative for Stroke). Max Confidence: {1-max_prob:.1%}")
        print("🛑 Halting pipeline. No clinical segmentation required.")
        print("="*60)
        return

    print(f"⚠️ DIAGNOSIS: STROKE DETECTED. Peak Confidence: {max_prob:.1%}")
    print("⏳ Stage 2: Initiating Volumetric Infarct Segmentation...")

    # --- STAGE 2: SEGMENTATION ---
    voxel_volume_cm3 = 0.00067  # User's predefined voxel scale
    total_infarct_pixels = 0

    # Pre-load all windowed slices into memory for z-axis stacking
    slices = []
    seg_img_size = (256, 256) # Define a consistent size for segmentation
    for img_name in image_files:
        img_path = os.path.join(patient_dir, img_name)
        windowed_img = apply_bone_window(img_path)
        if windowed_img is not None:
            # Resize the image to a fixed size compatible with DynUNet
            resized_img = cv2.resize(windowed_img, seg_img_size, interpolation=cv2.INTER_AREA)
            slices.append(resized_img)

    with torch.no_grad():
        # Iterate over each slice, pulling its neighbors into a 5-channel stack
        for idx in range(len(slices)):
            stack = []
            for offset in [-2, -1, 0, 1, 2]:
                n_idx = max(0, min(idx + offset, len(slices) - 1))
                stack.append(slices[n_idx])

            # Combine 5 contiguous slices and normalize 0-1
            z_stack = np.stack(stack, axis=-1).astype(np.float32) / 255.0
            z_stack = (z_stack - 0.5) / 0.5

            # Prepare tensor shape: (Batch, Channels, Height, Width) -> (1, 5, H, W)
            tensor_stack = torch.tensor(z_stack).permute(2, 0, 1).unsqueeze(0).to(device)

            # Forward pass through Volumetric DynUNet
            logits = segmenter(tensor_stack)
            probs = torch.sigmoid(logits)

            # We use 0.3 threshold to capture 'Total Affected Zone' (Penumbra + Core)
            preds = (probs > 0.3).float()
            total_infarct_pixels += preds.sum().item()

    total_volume_ml = total_infarct_pixels * voxel_volume_cm3

    print("="*60)
    print(f"🚨 FINAL RADIOLOGY REPORT - PATIENT {os.path.basename(patient_dir)} 🚨")
    print(f"Status: ACUTE INFARCT CONFIRMED")
    print(f"Total Infarct Pixels: {int(total_infarct_pixels)}")
    print(f"Estimated Infarct Volume: {total_volume_ml:.2f} mL")
    print("="*60)


---
## Execute Pipeline
Pass a directory containing patient CT slices to run the full diagnostic engine.

**Note for Hackathon Evaluation:** In the validation dataset, Stroke patients have dedicated segmentation folders, but Normal patients do not (they are grouped in the Classification dataset). The helper function below automatically tracks down the patient's images to test the pipeline properly.



In [ ]:
def get_patient_images_for_testing(patient_id):
    import shutil

    # 1. Stroke Patients usually have a dedicated folder in Segmentation dataset
    seg_folder = f"/content/drive/MyDrive/AI-Based_Infarct_Volume_Estimation_from_Non-Contrast_CT_for_Acute_Anterior_Circulation_Stroke/Segmentation/val/images/{patient_id}"
    if os.path.exists(seg_folder):
        return seg_folder

    # # 2. Normal Patients do not have a segmentation folder, their images are in the Classification dataset
    print(f"⚠️ No segmentation folder found for Patient {patient_id}.")
    class_normal_folder = "/content/drive/MyDrive/AI-Based_Infarct_Volume_Estimation_from_Non-Contrast_CT_for_Acute_Anterior_Circulation_Stroke/classification/val/Normal"

    if not os.path.exists(class_normal_folder):
        print(f"❌ Cannot find Classification dataset path: {class_normal_folder}")
        return None

    # Create a temporary folder to simulate a raw patient upload (for 100% realistic inference test)
    temp_dir = f"/tmp/patient_{patient_id}"
    os.makedirs(temp_dir, exist_ok=True)

    # Copy all slices belonging to patient_id exactly
    # Matches '001 01.png', '001_01.png', but strictly not '0010 01.png'
    found_any = False

    # Try different ID representations (exact match before space/underscore)
    for filename in os.listdir(class_normal_folder):
        name_part = filename.split('.')[0]
        # Split by non-digits
        parts = re.split(r'[_ -]', filename)

        # If the first part matches the patient_id exactly
        if parts[0] == str(patient_id):
            shutil.copy(os.path.join(class_normal_folder, filename), os.path.join(temp_dir, filename))
            found_any = True

    if found_any:
        return temp_dir
    else:
        print(f"❌ Could not find any images for Patient {patient_id} in Classification Normal dataset.")
        return None


In [ ]:
def get_patient_images_for_testing(patient_id):
    import shutil

    # 1. Stroke Patients usually have a dedicated folder in Segmentation dataset
    seg_folder = f"/content/drive/MyDrive/AI-Based_Infarct_Volume_Estimation_from_Non-Contrast_CT_for_Acute_Anterior_Circulation_Stroke/Segmentation/val/images/{patient_id}"
    if os.path.exists(seg_folder):
        return seg_folder

    # 2. Normal Patients do not have a segmentation folder, their images are in the Classification dataset
    # print(f"⚠️ No segmentation folder found for Patient {patient_id}. Assuming Normal patient and pulling from Classification Dataset...")
    class_normal_folder = "/content/drive/MyDrive/AI-Based_Infarct_Volume_Estimation_from_Non-Contrast_CT_for_Acute_Anterior_Circulation_Stroke/classification/val/Normal"

    if not os.path.exists(class_normal_folder):
        print(f"❌ Cannot find Classification dataset path: {class_normal_folder}")
        return None

    # Create a temporary folder to simulate a raw patient upload (for 100% realistic inference test)
    temp_dir = f"/tmp/patient_{patient_id}"
    os.makedirs(temp_dir, exist_ok=True)

    # Copy all slices starting with patient_id (e.g., '001 01.png', '001 02.png')
    found_any = False
    for filename in os.listdir(class_normal_folder):
        if filename.startswith(str(patient_id)):
            shutil.copy(os.path.join(class_normal_folder, filename), os.path.join(temp_dir, filename))
            found_any = True

    if found_any:
        return temp_dir
    else:
        print(f"❌ Could not find any images for Patient {patient_id} in Classification Normal dataset.")
        return None

In [ ]:
test_patient_id = "103"
patient_scan_directory = get_patient_images_for_testing(test_patient_id)

if patient_scan_directory:
    analyze_patient_ct(patient_scan_directory)

🔍 INITIATING ANALYSIS FOR PATIENT: patient_103
⏳ Stage 1: Running Rapid Classification Scan...
✅ DIAGNOSIS: NORMAL (Negative for Stroke). Max Confidence: 71.5%
🛑 Halting pipeline. No clinical segmentation required.


In [ ]:
test_patient_id = "093"
patient_scan_directory = get_patient_images_for_testing(test_patient_id)

if patient_scan_directory:
    analyze_patient_ct(patient_scan_directory)

🔍 INITIATING ANALYSIS FOR PATIENT: 093
⏳ Stage 1: Running Rapid Classification Scan...
⚠️ DIAGNOSIS: STROKE DETECTED. Peak Confidence: 99.3%
⏳ Stage 2: Initiating Volumetric Infarct Segmentation...
🚨 FINAL RADIOLOGY REPORT - PATIENT 093 🚨
Status: ACUTE INFARCT CONFIRMED
Total Infarct Pixels: 4174
Estimated Infarct Volume: 2.80 mL
